# Module 2.1: From Similarity Search to Connected Context

Use semantic search to find the right source, then traverse the graph to return compact, connected facts with provenance.

This notebook compares visible evidence for semantic entry, exact-term support, graph expansion, and structured filtering. A supporting Text2Cypher example appears after the deterministic exercises. Generated answer wording is outside this module's completion gate.

## Before you run this notebook

From the repository root or `notebooks/02-connected-context/`, prepare the deterministic 30-document lite graph:

```bash
cd notebooks/02-connected-context
uv run prepare_graph.py --mode lite
```

The command guarantees every source used below, reuses the pinned Amazon Nova embeddings on `:Chunk.embedding`, and provisions both retrieval indexes. It is safe to rerun.

In [ ]:
import os
import sys
from pathlib import Path


def locate_notebooks_root():
    override = os.environ.get('WORKSHOP_NOTEBOOKS_DIR')
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / 'workshop').is_dir():
            return candidate
        raise RuntimeError('WORKSHOP_NOTEBOOKS_DIR must contain the workshop package')

    starting_path = Path.cwd().resolve()
    for candidate in (starting_path, *starting_path.parents):
        if candidate.name == 'notebooks' and (candidate / 'workshop').is_dir():
            return candidate
        nested = candidate / 'notebooks'
        if (nested / 'workshop').is_dir():
            return nested
    raise RuntimeError(
        'Could not locate notebooks/workshop. Set WORKSHOP_NOTEBOOKS_DIR.'
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))
print(f'Workshop root: {REPO_ROOT}')

In [ ]:
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase, Query, READ_ACCESS
from neo4j_graphrag.retrievers import (
    HybridRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from neo4j_graphrag.retrievers.text2cypher import extract_cypher

load_dotenv(NOTEBOOKS_ROOT / '.env')
load_dotenv(REPO_ROOT / '.env')

from workshop.aws_region import aws_region, configure_aws_region
from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM
from workshop.graph_connection import (
    graph_database,
    neo4j_auth,
    neo4j_uri,
    require_neo4j_env,
)
from workshop.graph_schema import GRAPH_SCHEMA
from workshop.hybrid_retrieval import (
    GRAPH_QUERY_EXAMPLES,
    GRAPH_QUERY_PROMPT,
    pinned_schema_text,
    search_hotel_knowledge,
)
from workshop.retrieval_contract import (
    CHUNK_FULLTEXT_INDEX,
    CHUNK_VECTOR_INDEX,
    EMBEDDING_DIMENSIONS,
)
from workshop.retrieval_setup import (
    CHICAGO_EXCLUSION,
    CHICAGO_FILTER_QUERY,
    CHICAGO_QUALIFIER,
    CHICAGO_SOURCE_FILES,
    chicago_filter_problems,
    chicago_filter_records,
    fixture_problems,
    source_fixture_problems,
    verify_retrieval_indexes,
)

configure_aws_region()
require_neo4j_env()
DATABASE = graph_database()
driver = GraphDatabase.driver(neo4j_uri(), auth=neo4j_auth())
driver.verify_connectivity()
print(f'Connected to Neo4j database: {DATABASE}')

## Verify the prepared graph

Retrieval evidence is meaningful only when the source, embedding, entity, relationship, and index contracts are present. This readiness gate uses the shared checks for the exact Cairo and Chicago fixtures used below.

In [ ]:
try:
    verify_retrieval_indexes(driver)
    problems = fixture_problems(driver)
    problems.extend(source_fixture_problems(driver))
    chicago_records = chicago_filter_records(driver)
    problems.extend(chicago_filter_problems(chicago_records))
    if problems:
        raise RuntimeError('; '.join(problems))
except Exception as exc:
    raise RuntimeError(
        f'Module 2.1 is not ready: {exc}\n'
        'Run: cd notebooks/02-connected-context && uv run prepare_graph.py --mode lite'
    ) from exc

print(f'PASS  {CHUNK_VECTOR_INDEX}: online, cosine, {EMBEDDING_DIMENSIONS} dimensions')
print(f'PASS  {CHUNK_FULLTEXT_INDEX}: online over Chunk.text')
print('PASS  Cairo and Chicago source, path, field, amenity, and filter fixtures')

## The pinned graph and embedding contracts

The extraction pipeline and every traversal below use the shared `GRAPH_SCHEMA`. Every text retriever uses the same 1024-dimensional Amazon Nova embedding contract that wrote the chunk vectors. Every database call targets the configured Neo4j database.

In [ ]:
pattern_rows = ''.join(
    f'<tr><td><strong>{source}</strong></td><td>-[:{relationship}]-&gt;</td>'
    f'<td><strong>{target}</strong></td></tr>'
    for source, relationship, target in GRAPH_SCHEMA['patterns']
)
display(HTML(
    '<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>'
    f'</thead><tbody>{pattern_rows}</tbody></table>'
    '<p>Text provenance: <code>(:Chunk)-[:FROM_DOCUMENT]-&gt;(:Document)</code>.</p>'
    '<p>Entity provenance: <code>(:Hotel)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>'
))

## Evidence display helpers

Each retrieval block displays the question, retriever, configuration, top-k, rank, score, source filename, complete `Chunk.text`, approximate context size, and missing requested fields. Source filenames come from the graph provenance path.

In [ ]:
with driver.session(database=DATABASE, default_access_mode=READ_ACCESS) as session:
    chunk_sources = [
        record.data()
        for record in session.run(
            '''
            MATCH (chunk:Chunk)-[:FROM_DOCUMENT]->(document:Document)
            RETURN chunk.text AS chunk, document.source_filename AS source_filename
            '''
        )
    ]

source_by_chunk = {}
for row in chunk_sources:
    prior = source_by_chunk.setdefault(row['chunk'], row['source_filename'])
    assert prior == row['source_filename'], 'One chunk text resolves to multiple sources'


def text_result_formatter(record):
    node = record.get('node') or {}
    chunk = node.get('text') or ''
    return RetrieverResultItem(
        content=chunk,
        metadata={
            'score': record.get('score'),
            'source_filename': source_by_chunk.get(chunk),
        },
    )


embedder = BedrockEmbeddings(region_name=aws_region())

def evidence_rows(question, retriever_name, result, top_k, configuration, field_checks):
    print(f'Question: {question}')
    print(f'Retriever: {retriever_name}')
    print(f'Configuration: {configuration}')
    print(f'Top-k: {top_k}')
    print(f'Result count: {len(result.items)}')
    rows = []
    for rank, item in enumerate(result.items, 1):
        metadata = item.metadata or {}
        chunk = str(item.content or '')
        missing = [
            name for name, check in field_checks.items()
            if not check(chunk, metadata)
        ]
        row = {
            'rank': rank,
            'score': metadata.get('score'),
            'source_filename': metadata.get('source_filename'),
            'approx_context_chars': len(chunk),
            'missing_requested_fields': missing,
            'chunk': chunk,
        }
        rows.append(row)
        score = row['score']
        score_text = 'n/a' if score is None else f'{score:.6f}'
        print(f'Rank {rank} | score={score_text} | source={row["source_filename"]}')
        print(f'Approximate context: {row["approx_context_chars"]} characters')
        print(f'Missing requested fields: {missing or "none"}')
        print('Complete Chunk text:')
        print(chunk)
        print()
    return rows

## Pattern 1: semantic entry with VectorRetriever

The question paraphrases check-in as arrival processing. Vector retrieval should find the Cairo source without copying its heading or the phrase `Standard check-in time`. The completion gate checks retrieved evidence only.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)
ARRIVAL_QUESTION = (
    'When does standard arrival processing begin at AnyCompany Cairo Nile View?'
)
VECTOR_TOP_K = 3
arrival_result = vector_retriever.search(
    query_text=ARRIVAL_QUESTION,
    top_k=VECTOR_TOP_K,
)
arrival_rows = evidence_rows(
    ARRIVAL_QUESTION,
    'VectorRetriever',
    arrival_result,
    VECTOR_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, cosine, Nova {EMBEDDING_DIMENSIONS} dimensions',
    {
        'source_filename': lambda text, metadata: bool(metadata.get('source_filename')),
        'supported_arrival_time': lambda text, metadata: '3:00 PM' in text,
    },
)
cairo_arrival = [
    row for row in arrival_rows
    if row['source_filename'] == 'hotel-cairo-001.txt'
]
assert cairo_arrival, 'The Cairo source must appear in the top three vector results'
assert '3:00 PM' in cairo_arrival[0]['chunk']
print('PASS  Cairo source and supported 3:00 PM arrival time are visible.')

## Pattern 2: exact-term support with HybridRetriever

Postal code `60611` is a strong lexical signal and a weak semantic signal. Compare the same top-five limit for vector and hybrid retrieval. Hybrid uses the complete question as its vector signal, `60611` as its full-text signal, and the reviewed linear blend with `alpha=0.2`.

In [ ]:
IDENTIFIER_QUESTION = 'What is the cancellation policy for the hotel at 60611?'
IDENTIFIER_TOP_K = 5
vector_identifier_result = vector_retriever.search(
    query_text=IDENTIFIER_QUESTION,
    top_k=IDENTIFIER_TOP_K,
)

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=CHUNK_VECTOR_INDEX,
    fulltext_index_name=CHUNK_FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)
hybrid_identifier_result = hybrid_retriever.search(
    query_text='60611',
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=IDENTIFIER_TOP_K,
    ranker='linear',
    alpha=0.2,
)

identifier_checks = {
    'source_filename': lambda text, metadata: bool(metadata.get('source_filename')),
    'postal_code_60611': lambda text, metadata: '60611' in text,
    'cancellation_policy': lambda text, metadata: (
        'at least 24 hours prior to arrival' in text.casefold()
    ),
}
vector_identifier_rows = evidence_rows(
    IDENTIFIER_QUESTION,
    'VectorRetriever',
    vector_identifier_result,
    IDENTIFIER_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, semantic signal=complete question',
    identifier_checks,
)
hybrid_identifier_rows = evidence_rows(
    IDENTIFIER_QUESTION,
    'HybridRetriever',
    hybrid_identifier_result,
    IDENTIFIER_TOP_K,
    'linear ranker, alpha=0.2, vector signal=complete question, full-text term=60611',
    identifier_checks,
)
for label, rows in (('Vector', vector_identifier_rows), ('Hybrid', hybrid_identifier_rows)):
    for row in rows:
        row['exact_term_hits'] = ['60611'] if '60611' in row['chunk'] else []
        print(
            f"{label} rank {row['rank']} | source={row['source_filename']} | "
            f"exact_term_hits={row['exact_term_hits']}"
        )

windward_hybrid = [
    row for row in hybrid_identifier_rows
    if row['source_filename'] == 'hotel-chicago-001.txt'
]
assert windward_hybrid, 'Hybrid must return hotel-chicago-001.txt in its top five'
windward_chunk = windward_hybrid[0]['chunk']
assert 'Windward Mile Tower' in windward_chunk
assert '60611' in windward_chunk
assert 'at least 24 hours prior to arrival' in windward_chunk.casefold()
vector_rank_scan = vector_retriever.search(
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=30,
)
live_vector_rank = next(
    (
        rank
        for rank, item in enumerate(vector_rank_scan.items, 1)
        if (item.metadata or {}).get('source_filename') == 'hotel-chicago-001.txt'
    ),
    None,
)
rank_text = str(live_vector_rank) if live_vector_rank is not None else 'outside top 30'
print(f'Live vector rank for hotel-chicago-001.txt: {rank_text}')
print('PASS  Hybrid evidence contains the hotel, postal code, and cancellation policy.')

## Pattern 3: semantic entry plus connected facts with VectorCypherRetriever

Vector retrieval finds a relevant source. Vector-Cypher starts from that semantic match and follows reviewed relationships to named hotel fields and authored amenities. These graph fields reflect what extraction placed in Neo4j, not an independent source of truth. The record keeps the source `Chunk`, source `Document`, semantic score, and provenance paths visible so you can inspect omissions or merges against the authored source.

In [ ]:
VECTOR_CYPHER_QUERY = '''
MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)-[:FROM_DOCUMENT]->(document:Document)
CALL (hotel) {
    MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity)
    WHERE amenity.name IS NOT NULL
    WITH DISTINCT amenity.name AS amenity_name
    ORDER BY toLower(amenity_name)
    RETURN collect(amenity_name) AS amenities
}
RETURN hotel.name AS hotel_name,
       hotel.hotel_id AS hotel_id,
       hotel.guest_rating AS guest_rating,
       document.source_filename AS source_filename,
       amenities,
       node.text AS source_chunk,
       score AS semantic_score,
       ['FROM_DOCUMENT', 'FROM_CHUNK', 'OFFERS_AMENITY'] AS relationship_types,
       {
           source_chunk: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           source_filename: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           hotel_name: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           hotel_id: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           guest_rating: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           amenities: '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)'
       } AS field_provenance
ORDER BY semantic_score DESC, hotel_id ASC NULLS LAST
'''

def graph_result_formatter(record):
    requested = (
        'hotel_name',
        'hotel_id',
        'guest_rating',
        'source_filename',
        'amenities',
    )
    metadata = {
        'hotel_name': record.get('hotel_name'),
        'hotel_id': record.get('hotel_id'),
        'guest_rating': record.get('guest_rating'),
        'source_filename': record.get('source_filename'),
        'amenities': record.get('amenities') or [],
        'semantic_score': record.get('semantic_score'),
        'relationship_types': record.get('relationship_types') or [],
        'field_provenance': record.get('field_provenance') or {},
    }
    metadata['missing_requested_fields'] = [
        field for field in requested
        if metadata.get(field) is None or metadata.get(field) == []
    ]
    return RetrieverResultItem(
        content=record.get('source_chunk') or '',
        metadata=metadata,
    )

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    retrieval_query=VECTOR_CYPHER_QUERY,
    embedder=embedder,
    result_formatter=graph_result_formatter,
    neo4j_database=DATABASE,
)
CAIRO_GRAPH_QUESTION = (
    'What amenities and guest rating does AnyCompany Cairo Nile View have?'
)
GRAPH_TOP_K = 3
graph_vector_result = vector_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)
graph_vector_rows = evidence_rows(
    CAIRO_GRAPH_QUESTION,
    'VectorRetriever',
    graph_vector_result,
    GRAPH_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, semantic entry before graph expansion',
    {
        'source_filename': lambda text, metadata: bool(metadata.get('source_filename')),
        'hotel_name': lambda text, metadata: 'AnyCompany Cairo Nile View' in text,
        'hotel_id': lambda text, metadata: False,
        'guest_rating': lambda text, metadata: '4.5/5.0' in text,
        'amenities': lambda text, metadata: 'Hotel Amenities' in text,
    },
)
graph_result = vector_cypher_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)
graph_records = []
print(f'Question: {CAIRO_GRAPH_QUESTION}')
print('Retriever: VectorCypherRetriever')
print(f'Configuration: index={CHUNK_VECTOR_INDEX}, top_k={GRAPH_TOP_K}, reviewed traversal')
print(f'Result count: {len(graph_result.items)}')
for rank, item in enumerate(graph_result.items, 1):
    metadata = dict(item.metadata or {})
    metadata['source_chunk'] = str(item.content or '')
    metadata['rank'] = rank
    metadata['approx_context_chars'] = len(metadata['source_chunk']) + len(str(metadata))
    graph_records.append(metadata)
    print(f'Record {rank}:')
    for field in (
        'hotel_name',
        'hotel_id',
        'guest_rating',
        'source_filename',
        'amenities',
        'semantic_score',
        'relationship_types',
        'field_provenance',
        'missing_requested_fields',
        'approx_context_chars',
    ):
        print(f'  {field}: {metadata[field]}')
    print('  source_chunk:')
    print(metadata['source_chunk'])

cairo_graph = next(
    record for record in graph_records
    if record['source_filename'] == 'hotel-cairo-001.txt'
)
assert cairo_graph['hotel_name'] == 'AnyCompany Cairo Nile View'
assert cairo_graph['hotel_id'] == '81393d51-1df3-4f53-b58e-e4cda9736fd7'
assert cairo_graph['guest_rating'] == 4.5
assert not cairo_graph['missing_requested_fields']
assert cairo_graph['semantic_score'] is not None
assert set(cairo_graph['relationship_types']) == {
    'FROM_DOCUMENT',
    'FROM_CHUNK',
    'OFFERS_AMENITY',
}
for field in (
    'source_chunk',
    'source_filename',
    'hotel_name',
    'hotel_id',
    'guest_rating',
    'amenities',
):
    assert field in cairo_graph['field_provenance']
amenity_text = ' '.join(cairo_graph['amenities']).casefold()
for required_term in ('pool', 'spa', 'fitness', 'wifi', 'restaurant'):
    assert required_term in amenity_text, required_term

vector_cairo = next(
    row for row in graph_vector_rows
    if row['source_filename'] == 'hotel-cairo-001.txt'
)
requested_graph_fields = {
    'hotel_name',
    'hotel_id',
    'guest_rating',
    'source_filename',
    'amenities',
}
vector_named_fields = {'source_filename'}
graph_named_fields = requested_graph_fields - set(cairo_graph['missing_requested_fields'])
print('Evidence comparison:')
print(
    f'  Vector: {len(vector_named_fields)}/{len(requested_graph_fields)} named fields, '
    f"about {vector_cairo['approx_context_chars']} context characters"
)
print(
    f'  Vector-Cypher: {len(graph_named_fields)}/{len(requested_graph_fields)} named fields, '
    f"about {cairo_graph['approx_context_chars']} context characters"
)
print('Extraction quality limits graph enrichment. Missing extracted relationships stay missing.')
print('PASS  Cairo Vector-Cypher evidence includes every locked field and provenance path.')

## Pattern 4: reviewed structured AND filtering

The required structured demonstration uses the shared fixed Cypher contract. It evaluates both amenity predicates against the same hotel and returns records, rather than a pool count. This makes the candidate set, qualifier, and exclusion visible.

In [ ]:
CHICAGO_QUESTION = 'Which hotels in Chicago offer both a spa and a swimming pool?'
candidate_records = chicago_filter_records(driver)
qualifying_records = [row for row in candidate_records if row['qualifies']]
excluded_records = [row for row in candidate_records if not row['qualifies']]

print(f'Question: {CHICAGO_QUESTION}')
print('Purpose: require connected spa and swimming-pool amenities on the same hotel')
print(f'Reviewed Cypher: {CHICAGO_FILTER_QUERY}')
print(f'Parameters: source_filenames={list(CHICAGO_SOURCE_FILES)}')
print(f'Candidate count: {len(candidate_records)}')
print(f'Qualifier count: {len(qualifying_records)}')
print(f'Exclusion count: {len(excluded_records)}')
print('Candidate records:')
for record in candidate_records:
    print(record)
print('Qualifying records:')
for record in qualifying_records:
    print(record)
print('Excluded records:')
for record in excluded_records:
    print(record)

assert not chicago_filter_problems(candidate_records)
assert {row['source_filename'] for row in candidate_records} == set(CHICAGO_SOURCE_FILES)
assert [row['hotel_name'] for row in qualifying_records] == [CHICAGO_QUALIFIER]
assert len(excluded_records) == 1
assert excluded_records[0]['hotel_name'] == CHICAGO_EXCLUSION
assert set(excluded_records[0]['missing_required_amenities']) == {
    'spa',
    'swimming pool',
}
print('PASS  Two candidates, one qualifier, and the Windward exclusion are explicit.')

## Optional support: Text2Cypher behind a trust boundary

Generated Cypher is useful for flexible structured questions, but it is outside the deterministic completion gate. This optional cell requires distinct `NEO4J_READ_USERNAME` and `NEO4J_READ_PASSWORD` credentials whose current Neo4j role is `reader`. If that contract is unavailable, the cell reports a blocked state and executes no generated query. It uses the shared model and pinned schema, plans with `EXPLAIN`, targets the configured database, and gives each database query a 15-second timeout. It displays the query, validation state, records, and errors.

A production connection should also use a read-only Neo4j user. The database then enforces the same boundary independently.

In [ ]:
TEXT2CYPHER_TIMEOUT_SECONDS = 15


def run_optional_text2cypher(question):
    outcome = {
        'question': question,
        'generated_cypher': '',
        'read_only_validation': 'not planned',
        'records': [],
        'result_count': 0,
        'execution_error': None,
    }
    read_driver = None
    try:
        read_username = os.environ.get('NEO4J_READ_USERNAME')
        read_password = os.environ.get('NEO4J_READ_PASSWORD')
        if not read_username or not read_password:
            outcome['read_only_validation'] = (
                'blocked: set NEO4J_READ_USERNAME and NEO4J_READ_PASSWORD'
            )
            raise RuntimeError('Distinct read-only Neo4j credentials are required')

        read_driver = GraphDatabase.driver(
            neo4j_uri(),
            auth=(read_username, read_password),
        )
        with read_driver.session(
            database=DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            current_user = session.run(
                Query(
                    'SHOW CURRENT USER YIELD user, roles RETURN user, roles',
                    timeout=TEXT2CYPHER_TIMEOUT_SECONDS,
                )
            ).single(strict=True)
        roles = {role.casefold() for role in current_user['roles']}
        write_roles = {'admin', 'architect', 'publisher', 'editor'}
        if 'reader' not in roles or roles & write_roles:
            outcome['read_only_validation'] = (
                f'blocked: roles={sorted(roles)} do not satisfy reader-only contract'
            )
            raise RuntimeError('Neo4j credentials are not reader-only')
        outcome['read_only_validation'] = 'passed: database role=reader'

        prompt = GRAPH_QUERY_PROMPT.format(
            schema=pinned_schema_text(),
            examples=' '.join(GRAPH_QUERY_EXAMPLES),
            query_text=question,
        )
        response = BedrockLLM(region_name=aws_region()).invoke(prompt)
        cypher = extract_cypher(response.content)
        outcome['generated_cypher'] = cypher

        with read_driver.session(
            database=DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            summary = session.run(
                Query(f'EXPLAIN {cypher}', timeout=TEXT2CYPHER_TIMEOUT_SECONDS)
            ).consume()
            if summary.query_type != 'r':
                outcome['read_only_validation'] = (
                    f'rejected: query_type={summary.query_type}'
                )
                raise RuntimeError('The planner did not classify the query as read-only')
            outcome['read_only_validation'] = (
                'passed: database role=reader and EXPLAIN query_type=r'
            )
            outcome['records'] = session.run(
                Query(cypher, timeout=TEXT2CYPHER_TIMEOUT_SECONDS)
            ).data()[:25]
            outcome['result_count'] = len(outcome['records'])
    except Exception as exc:
        outcome['execution_error'] = f'{type(exc).__name__}: {exc}'
    finally:
        if read_driver is not None:
            read_driver.close()
    return outcome


text2cypher_outcome = run_optional_text2cypher(CHICAGO_QUESTION)
for field, value in text2cypher_outcome.items():
    print(f'{field}: {value}')
print('Supporting Text2Cypher output shown. Fixed Cypher remains the acceptance path.')

## Choose a retrieval pattern

| Query shape | Start with | Evidence to inspect | Role in this workshop |
|---|---|---|---|
| Semantic paraphrase | `VectorRetriever` | Ranked `Chunk` nodes, vector scores, source provenance | Core semantic entry pattern |
| Exact term plus semantic intent | `HybridRetriever` | Exact-term hits beside ranked `Chunk` nodes | Supporting exact-term pattern |
| Semantic entry plus named connected facts | `VectorCypherRetriever` | Source `Chunk`, graph fields, relationships, provenance | Core connected-context pattern |
| Stable structured constraint | Reviewed fixed Cypher | Candidate, qualifier, and exclusion records | Deterministic structured pattern |
| Flexible structured question | Text2Cypher | Generated query, planner state, records, errors | Optional governed interface |
| Application retrieval in Module 3 | `search_hotel_knowledge` | Hybrid-Cypher evidence records | Selected handoff pattern |

`HybridCypherRetriever` combines exact-term support with reviewed graph expansion. Module 2 selects it because the booking question needs an exact hotel name and connected named fields in one evidence record. The application freezes that combination behind `search_hotel_knowledge` so Module 3 can focus on grounding, abstention, and protected reservation writes.

## Chunking note

This workshop uses large chunks during graph extraction so each hotel remains intact while the pipeline creates entities and relationships. A production system can retain that extraction strategy and create separate, smaller retrieval chunks. Choose retrieval chunk size from the source material, expected questions, and evidence budget.

In [ ]:
selected_module_3_retriever = search_hotel_knowledge
assert selected_module_3_retriever.__name__ == 'search_hotel_knowledge'
print('Selected for Module 3: workshop.hybrid_retrieval.search_hotel_knowledge')
driver.close()
print('Connection closed.')